<a href="https://colab.research.google.com/github/jae0029/Data-Mining/blob/main/Copy_of_hw1a_design_and_test_a_statistic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 1A: Design and Test a Statistic

Statistical software can evaluate many statistics. It cannot decide which
statistic represents the question that matters. You will first use a supplied
statistic with SciPy, then design and test a different statistic for a new
operational question.

The original data come from the
[`palmerpenguins` project](https://github.com/allisonhorst/palmerpenguins/blob/main/inst/extdata/penguins.csv).
The version used here has been modified for this assignment. Use the provided
teaching extract rather than substituting the original file.

## Scenario and analytical setup

A field research program has a file of recorded penguin measurements. The data
team first wants to compare average body mass between the documented male and
female groups. Later, the field-operations team will ask a different question
about an equipment limit.

Read the accompanying data card before beginning. In one short response, state:

- what one row represents;
- which fields define the groups and body-mass measurement, including the unit;
  and
- one reason this file should not automatically be treated as a random sample
  of all penguins.

**Your response:**

> One row represents one recorded penguin observation. The groups are defined by the sex field, and body mass is measured by the body_mass_g field in grams. This file should not automatically be treated as a random sample of all penguins because the observations come from a specific research dataset collected from specific islands and times rather than from a documented random sample of the entire penguin population.

In [23]:
# pip install jupyter ipykernel numpy pandas matplotlib scipy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import permutation_test

penguins = pd.read_csv(
    "https://raw.githubusercontent.com/olearydj/INSY7130/"
    "main/homework/hw1a/data/penguin-mass-records.csv"
)
penguins.shape

(333, 8)

In [24]:
# penguins.columns
penguins.head()
# penguins.info()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181,3750,male,2007
1,Adelie,Torgersen,39.5,17.4,186,3800,female,2007
2,Adelie,Torgersen,40.3,18.0,195,3250,female,2007
3,Adelie,Torgersen,36.7,19.3,193,3450,female,2007
4,Adelie,Torgersen,39.3,20.6,190,3650,male,2007


The following check confirms whether the two required fields contain missing
values in this teaching extract.

In [25]:
penguins[["sex", "body_mass_g"]].isna().sum()

sex            0
body_mass_g    0
dtype: int64

Create `male_mass` and `female_mass` as one-dimensional NumPy arrays. Print the
number of observations in each array.

In [26]:
# Create male_mass and female_mass, then print their lengths.
male_mass = penguins.loc[
    penguins["sex"] == "male",
    "body_mass_g"
].to_numpy()

female_mass = penguins.loc[
    penguins["sex"] == "female",
    "body_mass_g"
].to_numpy()

print("Male observations:", len(male_mass))
print("Female observations:", len(female_mass))

Male observations: 168
Female observations: 165


## Part 1: Use a supplied statistic

The data team supplies the statistic

$$
T_{\text{mean}}
=
\overline{x}_{\text{male}}
-
\overline{x}_{\text{female}}.
$$

It returns one number measured in grams. A positive value means that the
recorded male group has the higher mean; a negative value means that the
recorded female group has the higher mean.

In [27]:
def mean_difference(male, female):
    return np.mean(male) - np.mean(female)


# Calculate and display the observed mean difference.
observed_mean_diff = mean_difference(male_mass, female_mass)

print(f"Observed mean difference: {observed_mean_diff:.2f} g")

Observed mean difference: 683.41 g


Before testing, explain the no-relationship model in the context of the
recorded sex labels and body-mass measurements. State what one independent
permutation breaks and what it preserves.

**Your response:**

> Under the no-relationship model, the body-mass measurements are not associated with the recorded sex labels. In this model, any body-mass value could have been assigned to either the male or female group, so the observed difference in mean body mass is due only to random assignment. One independent permutation breaks the original connection between a penguin's recorded sex and its body-mass measurement by randomly reassigning the labels. At the same time, it preserves the observed body-mass values and the number of observations in each group.

Use a permutation test to evaluate the no-relationship hypothesis using the
supplied statistic and samples. Use an independent, two-sided method with
5,000 resamples and 7130 as the random seed. Write the test call, store its
result as `mean_result`, and print the observed statistic and simulated
p-value.

In [28]:
# Create the random generator, write the permutation test, and print its result.
mean_result = permutation_test(
    data=(male_mass, female_mass),
    statistic=mean_difference,
    permutation_type="independent",
    alternative="two-sided",
    n_resamples=5000,
    random_state=7130
)

print(f"Observed statistic: {mean_result.statistic:.2f} g")
print(f"Simulated p-value: {mean_result.pvalue:.6f}")

Observed statistic: 683.41 g
Simulated p-value: 0.000400


Interpret the first test in one short paragraph. Report the observed statistic
with its direction and unit, explain the simulated p-value under the
no-relationship model, and state both a bounded conclusion and one conclusion
the test does not support. Do not describe the p-value as the probability that
the null hypothesis is true.

**Your interpretation:**

> The observed mean difference was 683.41 g, indicating that the recorded male penguins had a higher average body mass than the recorded female penguins by about 683 grams. Under the no-relationship model, where body mass and the recorded sex labels are unrelated, a difference this large or larger occurred in only about 0.04% of the 5,000 simulated permutations (p = 0.0004). This provides strong evidence that body mass and the recorded sex labels are associated in this dataset. However, this conclusion is limited to the observed data and supports only an association between sex and body mass. The test does not support a conclusion that sex causes differences in body mass or that the results automatically generalize to all penguins.

## Part 2: Design a statistic for the team's question

The field-operations team explains that average body mass does not directly
answer its question. Standard equipment can be used only for birds weighing at
most 4,500 g. The team wants to know whether the documented male and female
groups differ in how often recorded body mass exceeds that limit.

Design the statistic before running the test. It must return one number, use
the proportion exceeding 4,500 g within each group, be signed rather than
absolute, and use a subtraction order declared in advance.

```text
Outcome:
Subtraction order:
Statistic:
Positive / negative values mean:
```

**Your design:**

> 
```text
Outcome: Whether a penguin's recorded body mass exceeds 4,500 g.
Subtraction order: Male exceedance rate − Female exceedance rate.
Statistic: Proportion of males with body mass greater than 4,500 g minus the proportion of females with body mass greater than 4,500 g.
Positive / negative values mean: A positive value means the documented male group exceeds 4,500 g more often than the documented female group. A negative value means the documented female group exceeds 4,500 g more often than the documented male group.
```

Complete the function. Its inputs remain the two body-mass arrays. Inside the
function, calculate each within-group exceedance rate and return their signed
difference in your declared order.

In [29]:
EQUIPMENT_LIMIT_G = 4_500


def exceedance_rate_difference(male, female):
    # Calculate the two within-group exceedance rates.
    # Return their signed difference in your declared order.
    # pass
    male_rate = np.mean(male > EQUIPMENT_LIMIT_G)
    female_rate = np.mean(female > EQUIPMENT_LIMIT_G)
    return male_rate - female_rate

Calculate and display both observed exceedance rates and the signed difference.
Interpret the difference in percentage points. Create one clearly labeled plot
comparing the two rates.

In [30]:
# Calculate and display the rates and signed difference, then create the plot.
male_rate = np.mean(male_mass > EQUIPMENT_LIMIT_G)
female_rate = np.mean(female_mass > EQUIPMENT_LIMIT_G)

observed_exceedance_diff = exceedance_rate_difference(
    male_mass,
    female_mass
)

print(f"Male exceedance rate: {male_rate:.3f}")
print(f"Female exceedance rate: {female_rate:.3f}")
print(f"Signed difference: {observed_exceedance_diff:.3f}")

Male exceedance rate: 0.417
Female exceedance rate: 0.255
Signed difference: 0.162


Use a second independent, two-sided permutation test with the original mass
arrays, your new statistic, 5,000 resamples, and seed 7130. Store the result as
`exceedance_result` and print its statistic and p-value.

In [ ]:
# Create a new random generator, run the test, and print its result.
rng = np.random.default_rng(7130)

exceedance_result = permutation_test(
    data=(male_mass, female_mass),
    statistic=exceedance_rate_difference,
    permutation_type="independent",
    alternative="two-sided",
    n_resamples=5000,
    rng=rng
)

print(f"Observed statistic: {exceedance_result.statistic:.3f}")
print(f"Simulated p-value: {exceedance_result.pvalue:.6f}")

Observed statistic: 0.162
Simulated p-value: 0.003199


Interpret the operational test in one short paragraph. Report both group rates,
the signed percentage-point difference, and the simulated p-value. Explain the
p-value under the corresponding no-relationship model, give a bounded
conclusion, and name one limitation relevant to an equipment decision.

**Your interpretation:**

> The proportion of penguins with recorded body mass exceeding the 4,500 g equipment limit was 0.417 (41.7%) for the documented male group and 0.255 (25.5%) for the documented female group. The signed difference was 0.162, meaning that the male group exceeded the limit 16.2 percentage points more often than the female group. Under the no-relationship model, where body mass and the recorded sex labels are unrelated, a difference in exceedance rates at least this extreme occurred in about 0.32% of the 5,000 simulated permutations (p = 0.003199). This provides strong evidence that the frequency of exceeding the 4,500 g equipment limit differs between the documented male and female groups in this dataset. However, a limitation relevant to an equipment decision is that these results are based on a specific sample of penguins and may not fully represent the birds that would be encountered in future field operations.

## Reflection and disclosure

In two or three sentences, explain what remained the same between the tests,
what changed, and why the second statistic better represents the equipment
team's question.

**Your reflection:**

> Both tests used the same body-mass data, the same male and female groups, and the same independent two-sided permutation test procedure with 5,000 resamples. What changed was the statistic being tested: the first compared mean body mass, while the second compared the proportion of penguins exceeding the 4,500 g equipment limit. The second statistic better represents the equipment team's question because the equipment decision depends on whether birds exceed the weight threshold, not on the average body mass of each group.

Identify any material assistance and how you checked it, or write `None`.

**Assistance disclosure:**

> I used Microsoft Copilot to help understand permutation testing concepts, check Python code, and review the interpretation of statistical results. I verified the assistance by running all code myself, examining the notebook outputs, and ensuring that the written interpretations matched the displayed results.